# Emulating device noise

In this notebook, you will learn how to:

- fetch a device from a connection and use its noise model,
- emulate a program with the device noise model,
- compare ideal and noisy results, through occupations and bitstrings.

## Fetching a device with a noise model

Here, we fetch a device from PasqalCloudConnection which has a noise model.

In [ ]:
from pasqal_cloud import PasqalCloudConnection

from qoolqit import Device

device = Device.from_connection(connection=PasqalCloudConnection(), name="FRESNEL_CAN1")

## Rabi oscillations of a blockaded pair

We drive two qubits placed at the minimum distance of the device, where their interaction energy is $1$ in dimensionless units, with a constant amplitude $\Omega = 0.1$. 

Since the interaction is much larger than $\Omega$, the pair is in the blockade regime: at most one qubit can be excited, so the occupation of each qubit oscillates between $0$ and $1/2$. We drive for the longest duration allowed by the device, so that noise has time to build up.

In [ ]:
from qoolqit import ConstantWaveform, Drive, QuantumProgram, Register

omega = 0.1  # much smaller than the interaction energy (1.0 at distance 1): blockade regime
T = device.specs["max_duration"]  # longest drive allowed by the device

register = Register.from_coordinates([(0.0, 0.0), (1.0, 0.0)])
drive = Drive(amplitude=ConstantWaveform(T, omega))
program = QuantumProgram(register, drive)

program.compile_to(device)
print(program)

## Switching on the device noise

We ask the emulator for the Rydberg `Occupation` of each qubit along the pulse. Starting from an ideal configuration, the noisy one is obtained by setting `prefer_device_noise_model=True`. 

Since the device noise is stochastic, the emulation averages over `n_trajectories` random realizations of the noise.

In [ ]:
from qoolqit.execution import EmulationConfig, LocalEmulator, Occupation

rel_times = [i / 200 for i in range(201)]  # relative evaluation times, between 0 and 1
observables = (Occupation(evaluation_times=rel_times),)

ideal_config = EmulationConfig(observables=observables)
# same configuration, with the device noise switched on
noisy_config = EmulationConfig(
    observables=observables, prefer_device_noise_model=True, n_trajectories=64
)

ideal_results = LocalEmulator(emulation_config=ideal_config).run(program).results()
noisy_results = LocalEmulator(emulation_config=noisy_config).run(program).results()

In [ ]:
import matplotlib.pyplot as plt

# evaluation times are relative to the drive duration
times = [rel_t * T for rel_t in rel_times]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for qubit, ax in enumerate(axes):
    ax.plot(times, [occ[qubit] for occ in ideal_results.occupation], label="ideal")
    ax.plot(
        times,
        [occ[qubit] for occ in noisy_results.occupation],
        "-o",
        ms=4,
        markevery=4,
        label="noisy",
    )
    ax.set_title(f"qubit {qubit}")
    ax.set_xlabel("$t$")
    ax.legend(loc="upper right")
axes[0].set_ylabel("Rydberg occupation")
plt.show()

Without noise, the two qubits oscillate in phase with full contrast, between $0$ and $1/2$. With noise, the oscillation damps: the maxima decrease and the minima lift from zero.

Noise also shows up in the bitstrings sampled at the end of the pulse: the noisy counts deviate from the ideal ones, mostly because of state preparation and measurement (SPAM) errors. Note that the noisy counter collects the shots of every trajectory.

In [ ]:
print("Ideal:", ideal_results.final_bitstrings)
print("Noisy:", noisy_results.final_bitstrings)

print("Number of ideal results (n_shots):", sum(ideal_results.final_bitstrings.values()))
print(
    "Number of noisy results (n_trajectories x n_shots):",
    sum(noisy_results.final_bitstrings.values()),
)